In [1]:
# =========================
# Feature Overlap Analysis: Raw Proteins vs PCRR Ratios
# 
# For raw protein models: a protein is "predictive" if it has non-zero feature importance
# in at least 2 of 5 splits for a given class.
# 
# For ratio models: a ratio like SEMA3C_minus_APOE is "predictive" if it has non-zero
# feature importance in at least 2 of 5 splits. We then decompose each predictive ratio
# into its two constituent proteins (e.g., SEMA3C and APOE) and collect the unique set.
# 
# We then compare the overlap between the two protein sets per class.
# =========================
import os
import pickle
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# ---- Config ----
raw_model_dir = "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)_fixed"
ratio_model_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Protein_ratios_noRFE"

classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]
MIN_SPLITS = 2  # predictive if non-zero importance in at least this many splits

def safe_cls(c): return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

# =============================
# 1) Raw protein models: get predictive proteins per class
# =============================
raw_feature_counts_per_class = defaultdict(Counter)

for seed in seeds:
    for cls in classes:
        model_path = os.path.join(raw_model_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl")
        if not os.path.exists(model_path):
            print(f"Missing: {model_path}")
            continue

        with open(model_path, "rb") as f:
            automl = pickle.load(f)

        importances = automl.model.estimator.feature_importances_
        features = automl.feature_names_in_
        nonzero_features = [feat for feat, imp in zip(features, importances) if imp > 0]
        raw_feature_counts_per_class[cls].update(nonzero_features)

raw_predictive_proteins_per_class = {
    cls: set(feat for feat, count in counter.items() if count >= MIN_SPLITS)
    for cls, counter in raw_feature_counts_per_class.items()
}

# =============================
# 2) Ratio models: get predictive ratios per class, decompose into proteins
# =============================
ratio_feature_counts_per_class = defaultdict(Counter)

for seed in seeds:
    for cls in classes:
        model_path = os.path.join(ratio_model_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl")
        if not os.path.exists(model_path):
            print(f"Missing: {model_path}")
            continue

        with open(model_path, "rb") as f:
            automl = pickle.load(f)

        importances = automl.model.estimator.feature_importances_
        features = automl.feature_names_in_
        nonzero_features = [feat for feat, imp in zip(features, importances) if imp > 0]
        ratio_feature_counts_per_class[cls].update(nonzero_features)

# Get predictive ratios (>=2 splits), then decompose into constituent proteins
ratio_predictive_proteins_per_class = {}
ratio_predictive_ratios_per_class = {}

for cls in classes:
    counter = ratio_feature_counts_per_class[cls]
    predictive_ratios = [feat for feat, count in counter.items() if count >= MIN_SPLITS]
    ratio_predictive_ratios_per_class[cls] = predictive_ratios
    
    # Decompose each ratio "ProteinA_minus_ProteinB" into {ProteinA, ProteinB}
    proteins_from_ratios = set()
    for ratio_name in predictive_ratios:
        parts = ratio_name.split("_minus_")
        if len(parts) == 2:
            proteins_from_ratios.add(parts[0])
            proteins_from_ratios.add(parts[1])
        else:
            # Fallback in case naming convention differs
            print(f"WARNING: Could not parse ratio name: {ratio_name}")
    ratio_predictive_proteins_per_class[cls] = proteins_from_ratios

# =============================
# 3) Compare overlap per class
# =============================
print("=" * 80)
print(f"Feature Overlap Analysis (predictive = non-zero importance in >= {MIN_SPLITS} of {len(seeds)} splits)")
print("=" * 80)

for cls in classes:
    raw_prots = raw_predictive_proteins_per_class.get(cls, set())
    ratio_prots = ratio_predictive_proteins_per_class.get(cls, set())
    
    overlap = raw_prots & ratio_prots
    raw_only = raw_prots - ratio_prots
    ratio_only = ratio_prots - raw_prots
    union = raw_prots | ratio_prots
    
    jaccard = len(overlap) / len(union) if len(union) > 0 else 0
    
    print(f"\n--- {cls} ---")
    print(f"  Raw predictive proteins:          {len(raw_prots)}")
    print(f"  Ratio predictive ratios:          {len(ratio_predictive_ratios_per_class.get(cls, []))}")
    print(f"  Unique proteins from ratios:      {len(ratio_prots)}")
    print(f"  Overlap (in both):                {len(overlap)}")
    print(f"  Raw-only proteins:                {len(raw_only)}")
    print(f"  Ratio-only proteins:              {len(ratio_only)}")
    print(f"  Jaccard similarity:               {jaccard:.3f}")
    
    if overlap:
        print(f"  Overlapping proteins:             {sorted(overlap)}")

print("\n" + "=" * 80)
print("Summary Table")
print("=" * 80)
rows = []
for cls in classes:
    raw_prots = raw_predictive_proteins_per_class.get(cls, set())
    ratio_prots = ratio_predictive_proteins_per_class.get(cls, set())
    overlap = raw_prots & ratio_prots
    union = raw_prots | ratio_prots
    rows.append({
        "Class": cls,
        "Raw proteins (>=2/5)": len(raw_prots),
        "Ratio predictive ratios (>=2/5)": len(ratio_predictive_ratios_per_class.get(cls, [])),
        "Unique proteins from ratios": len(ratio_prots),
        "Overlap": len(overlap),
        "Jaccard": len(overlap) / len(union) if len(union) > 0 else 0,
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Note on runtime, it runs in about 5 minutes 11 seconds
# =========================
# Feature Importance Correlation: Raw Proteins vs PCRR Ratios
#
# 1) For each class, report how many of the total possible ratios (from the shortlisted
#    proteins) are actually predictive in >=2/5 splits.
#
# 2) For each shortlisted protein P:
#    - Raw importance: mean feature importance of P across all 5 splits in the raw model
#    - Ratio importance: among all ratios involving P, take the one with the highest mean
#      feature importance across all 5 splits (with zeros for splits where importance is 0).
#      Use that ratio's mean importance as the "ratio importance" for protein P.
#    Then compute Spearman rank correlation across all shortlisted proteins per class.
# =========================
import os
import pickle
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from itertools import combinations
from scipy.stats import spearmanr

# ---- Protein metadata: SeqId -> GeneSymbol mapping ----
protein_metadata_path = "/Users/adithyamadduri/Downloads/syn65414912/OhNM2025_ROSMAP_plasma_Soma7k_protein_metadata.csv"
df_prot_meta = pd.read_csv(protein_metadata_path)
seqid_to_gene = dict(zip(df_prot_meta["SeqId"], df_prot_meta["EntrezGeneSymbol"]))

def map_protein(seqid):
    return seqid_to_gene.get(seqid, seqid)

def map_ratio_name(name):
    if "_minus_" in name:
        left, right = name.split("_minus_")
        return f"{map_protein(left)}:{map_protein(right)}"
    return name

# ---- Config ----
raw_model_dir = "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)_fixed"
ratio_model_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Protein_ratios_noRFE"

classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]
MIN_SPLITS = 2

def safe_cls(c): return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

# =============================
# 1) Load raw model feature importances (per seed, per class)
# =============================
# Structure: raw_importances[cls][seed] = dict of {feature_name: importance}
raw_importances = defaultdict(dict)

for seed in seeds:
    for cls in classes:
        model_path = os.path.join(raw_model_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl")
        if not os.path.exists(model_path):
            print(f"Missing: {model_path}")
            continue
        with open(model_path, "rb") as f:
            automl = pickle.load(f)
        importances = automl.model.estimator.feature_importances_
        features = automl.feature_names_in_
        raw_importances[cls][seed] = dict(zip(features, importances))

# =============================
# 2) Get shortlisted proteins per class (>=2/5 splits with non-zero importance in raw model)
# =============================
raw_feature_counts_per_class = defaultdict(Counter)
for cls in classes:
    for seed in seeds:
        if seed not in raw_importances[cls]:
            continue
        for feat, imp in raw_importances[cls][seed].items():
            if imp > 0:
                raw_feature_counts_per_class[cls][feat] += 1

# Filter to proteins only (exclude demographic features like age_at_visit, educ, APOE_*)
demo_names = {"age_at_visit", "educ", "msex"}
shared_features_per_class = {
    cls: [feat for feat, count in counter.items()
          if count >= MIN_SPLITS and feat not in demo_names and not feat.startswith("APOE_")]
    for cls, counter in raw_feature_counts_per_class.items()
}

# =============================
# 3) Load ratio model feature importances (per seed, per class)
# =============================
ratio_importances = defaultdict(dict)

for seed in seeds:
    for cls in classes:
        model_path = os.path.join(ratio_model_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl")
        if not os.path.exists(model_path):
            print(f"Missing: {model_path}")
            continue
        with open(model_path, "rb") as f:
            automl = pickle.load(f)
        importances = automl.model.estimator.feature_importances_
        features = automl.feature_names_in_
        ratio_importances[cls][seed] = dict(zip(features, importances))

# =============================
# 4) Part 1: Ratio utilization — how many possible ratios are actually predictive?
# =============================
print("=" * 80)
print("Part 1: Ratio Utilization (of all possible ratios, how many are predictive >=2/5?)")
print("=" * 80)

# Count how many ratios are predictive (>=2/5 splits) per class
ratio_feature_counts_per_class = defaultdict(Counter)
for cls in classes:
    for seed in seeds:
        if seed not in ratio_importances[cls]:
            continue
        for feat, imp in ratio_importances[cls][seed].items():
            if imp > 0:
                ratio_feature_counts_per_class[cls][feat] += 1

for cls in classes:
    shortlist = shared_features_per_class[cls]
    n_proteins = len(shortlist)
    n_possible = n_proteins * (n_proteins - 1) // 2  # C(n, 2)
    n_predictive = sum(1 for feat, count in ratio_feature_counts_per_class[cls].items() if count >= MIN_SPLITS)
    pct = 100 * n_predictive / n_possible if n_possible > 0 else 0
    print(f"[{cls}] Shortlisted proteins: {n_proteins} | "
          f"Possible ratios C({n_proteins},2): {n_possible} | "
          f"Predictive ratios (>=2/5): {n_predictive} ({pct:.1f}%)")

# =============================
# 5) Part 2: Spearman correlation between raw protein importance and best-ratio importance
# =============================
print("\n" + "=" * 80)
print("Part 2: Spearman Rank Correlation (raw protein importance vs best-ratio importance)")
print("=" * 80)

for cls in classes:
    shortlist = shared_features_per_class[cls]
    if len(shortlist) < 3:
        print(f"[{cls}] Too few proteins for correlation. Skipping.")
        continue

    # --- Raw mean NORMALIZED importance per protein (averaged across all 5 splits, 0 if absent) ---
    # Normalization: per seed, divide each protein's importance by the total importance sum
    # for that seed, so importances sum to 1 within each seed. Then average across seeds.
    raw_mean_imp = {}
    for prot in shortlist:
        imps = []
        for seed in seeds:
            if seed in raw_importances[cls]:
                seed_imps = raw_importances[cls][seed]
                total = sum(seed_imps.values())
                if total > 0:
                    imps.append(seed_imps.get(prot, 0.0) / total)
                else:
                    imps.append(0.0)
            else:
                imps.append(0.0)
        raw_mean_imp[prot] = np.mean(imps)
    # --- Best-ratio mean importance per protein ---
    # For each protein P, find all ratios involving P, compute each ratio's mean importance
    # across all 5 splits (0 if not present), then take the max.
    
    # First, collect all ratio names from the ratio models for this class
    all_ratio_names = set()
    for seed in seeds:
        if seed in ratio_importances[cls]:
            all_ratio_names.update(ratio_importances[cls][seed].keys())

    # Compute mean NORMALIZED importance for each ratio across all 5 splits
    # Normalization: per seed, divide each ratio's importance by the total importance sum
    # for that seed's ratio model, so importances sum to 1 within each seed.
    ratio_mean_imp = {}
    for rname in all_ratio_names:
        imps = []
        for seed in seeds:
            if seed in ratio_importances[cls]:
                seed_imps = ratio_importances[cls][seed]
                total = sum(seed_imps.values())
                if total > 0:
                    imps.append(seed_imps.get(rname, 0.0) / total)
                else:
                    imps.append(0.0)
            else:
                imps.append(0.0)
        ratio_mean_imp[rname] = np.mean(imps)
    # For each protein, find its best ratio (highest mean importance among ratios involving it)
    best_ratio_imp = {}
    best_ratio_name = {}
    for prot in shortlist:
        # Find ratios involving this protein
        # Ratio names are like "ProtA_minus_ProtB"
        involving = []
        for rname, rimp in ratio_mean_imp.items():
            parts = rname.split("_minus_")
            if len(parts) == 2 and (parts[0] == prot or parts[1] == prot):
                involving.append((rname, rimp))
        
        if involving:
            best = max(involving, key=lambda x: x[1])
            best_ratio_imp[prot] = best[1]
            best_ratio_name[prot] = best[0]
        else:
            best_ratio_imp[prot] = 0.0
            best_ratio_name[prot] = None

    # --- Compute Spearman correlation ---
    proteins_with_both = [p for p in shortlist if p in raw_mean_imp and p in best_ratio_imp]
    raw_vals = np.array([raw_mean_imp[p] for p in proteins_with_both])
    ratio_vals = np.array([best_ratio_imp[p] for p in proteins_with_both])

    rho, pval = spearmanr(raw_vals, ratio_vals)
    print(f"\n[{cls}] n_proteins={len(proteins_with_both)} | Spearman rho={rho:.4f} | p={pval:.2e}")

    # --- Show top 10 proteins by raw importance with their best ratio ---
    ranked = sorted(proteins_with_both, key=lambda p: raw_mean_imp[p], reverse=True)
    print(f"  {'Protein':<15} {'Gene':<15} {'Raw Norm Imp':>14} {'Best Ratio Norm Imp':>20} {'Best Ratio (genes)':>35}")
    print(f"  {'-'*15} {'-'*15} {'-'*14} {'-'*20} {'-'*35}")
    for p in ranked[:10]:
        print(f"  {p:<15} {map_protein(p):<15} {raw_mean_imp[p]:>14.6f} {best_ratio_imp[p]:>20.6f} {map_ratio_name(str(best_ratio_name[p])):>35}")


/var/folders/yr/4bplyc1x1tq4xz4jckbg_8wr0000gn/T/ipykernel_91652/2419106933.py:59: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  automl = pickle.load(f)
/var/folders/yr/4bplyc1x1tq4xz4jckbg_8wr0000gn/T/ipykernel_91652/2419106933.py:59: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usag

Part 1: Ratio Utilization (of all possible ratios, how many are predictive >=2/5?)
[MCI] Shortlisted proteins: 43 | Possible ratios C(43,2): 903 | Predictive ratios (>=2/5): 812 (89.9%)
[NCI] Shortlisted proteins: 350 | Possible ratios C(350,2): 61075 | Predictive ratios (>=2/5): 100 (0.2%)
[AD] Shortlisted proteins: 155 | Possible ratios C(155,2): 11935 | Predictive ratios (>=2/5): 310 (2.6%)
[AD+] Shortlisted proteins: 23 | Possible ratios C(23,2): 253 | Predictive ratios (>=2/5): 51 (20.2%)

Part 2: Spearman Rank Correlation (raw protein importance vs best-ratio importance)

[MCI] n_proteins=43 | Spearman rho=0.0306 | p=8.46e-01
  Protein         Gene              Raw Norm Imp  Best Ratio Norm Imp                  Best Ratio (genes)
  --------------- --------------- -------------- -------------------- -----------------------------------
  10451-11        NUCB1                 0.036507             0.005940                         NUCB1:ERO1A
  14227-21        MYL6B                 0.